# Lab 04: Probabilistic Customer Segmentation and Segment Prediction
## Using Naive Bayes Classifiers on JanataHack Customer Segmentation Dataset

| Field | Value |
|---|---|
| **Registration** | 23MID0021 |
| **Name** | Geetha Priya S |
| **Course** | MDI3003 — Machine Learning |
| **Faculty** | Dr. Durgesh Kumar |
| **Lab** | 04 — Naive Bayes Customer Segmentation |

---


## 0. Imports and Setup

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder,
    LabelEncoder, KBinsDiscretizer
)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, CategoricalNB, ComplementNB
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, balanced_accuracy_score
)

warnings.filterwarnings('ignore')
np.random.seed(42)

FIG_DIR, MODEL_DIR, ART_DIR = 'figures', 'models', 'artifacts'
for d in [FIG_DIR, MODEL_DIR, ART_DIR]:
    os.makedirs(d, exist_ok=True)

print("Libraries loaded successfully.")


## 1. Feature Taxonomy

In [ ]:
FEATURE_COLS  = ['Gender','Ever_Married','Age','Graduated','Profession',
                 'Work_Experience','Spending_Score','Family_Size','Var_1']
TARGET_COL    = 'Segmentation'

NUMERIC_COLS  = ['Age','Work_Experience','Family_Size']
BINARY_COLS   = ['Gender','Ever_Married','Graduated']
ORDINAL_COLS  = ['Profession','Spending_Score','Var_1']
ALL_CAT_COLS  = BINARY_COLS + ORDINAL_COLS

FEATURE_GROUPS = {
    'Demographic':   ['Gender','Ever_Married','Age','Graduated','Profession'],
    'Psychographic': ['Spending_Score','Var_1'],
    'Behavioral':    ['Work_Experience','Family_Size'],
    'All Features':  FEATURE_COLS,
}
print("Feature groups:", list(FEATURE_GROUPS.keys()))


## 2. Custom Transformer — SafeOrdinalToNonNegative

In [ ]:
class SafeOrdinalToNonNegative(BaseEstimator, TransformerMixin):
    """Shifts feature matrix so all values are non-negative integers.
    Required by CategoricalNB which cannot accept negative category indices."""
    def fit(self, X, y=None):
        X = np.array(X, dtype=float)
        self.min_vals_ = np.nanmin(np.where(np.isnan(X), 0, X), axis=0)
        return self

    def transform(self, X):
        X = np.array(X, dtype=float)
        for col in range(X.shape[1]):
            nan_mask = np.isnan(X[:, col])
            if nan_mask.any():
                X[nan_mask, col] = self.min_vals_[col]
        shifts = np.minimum(self.min_vals_, 0)
        return (X - shifts).astype(int)


## 3. Load Dataset

In [ ]:
def generate_synthetic_dataset(n=8068, seed=42):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'ID': range(1, n+1),
        'Gender':          rng.choice(['Male','Female'], n, p=[0.56,0.44]),
        'Ever_Married':    rng.choice(['Yes','No',np.nan], n, p=[0.60,0.36,0.04]),
        'Age':             rng.integers(18, 90, n).astype(float),
        'Graduated':       rng.choice(['Yes','No',np.nan], n, p=[0.58,0.38,0.04]),
        'Profession':      rng.choice(
            ['Artist','Doctor','Engineer','Entertainment','Executive',
             'Healthcare','Homemaker','Lawyer','Marketing',np.nan], n,
            p=[0.108]*9 + [0.028]),
        'Work_Experience': rng.choice([float(i) for i in range(15)]+[np.nan], n,
                               p=[0.065]*14+[0.065,0.025]),
        'Spending_Score':  rng.choice(['Low','Average','High'], n, p=[0.33,0.34,0.33]),
        'Family_Size':     rng.choice([float(i) for i in range(1,10)]+[np.nan], n,
                               p=[0.11]*9+[0.01]),
        'Var_1':           rng.choice([f'Cat_{i}' for i in range(1,8)]+[np.nan], n,
                               p=[0.14]*7+[0.02]),
        'Segmentation':    rng.choice(['A','B','C','D'], n, p=[0.256,0.250,0.248,0.246]),
    })

for path in ['train.csv','data/train.csv']:
    if os.path.exists(path):
        df_raw = pd.read_csv(path)
        print(f"Loaded from '{path}'")
        break
else:
    print("train.csv not found — using synthetic data")
    df_raw = generate_synthetic_dataset()

print(f"Shape: {df_raw.shape}")
df_raw[FEATURE_COLS+[TARGET_COL]].head()


## 4. Exploratory Data Analysis

In [ ]:
print("Missing values:")
print(df_raw[FEATURE_COLS+[TARGET_COL]].isnull().sum())
print("\nClass distribution:")
print(df_raw[TARGET_COL].value_counts())


In [ ]:
df = df_raw[FEATURE_COLS+[TARGET_COL]].copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df[TARGET_COL].value_counts().sort_index()
axes[0].bar(counts.index, counts.values,
            color=['#2196F3','#FF9800','#4CAF50','#E91E63'], edgecolor='white')
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Count')

# Age by segment
for seg, col in zip(['A','B','C','D'],['#2196F3','#FF9800','#4CAF50','#E91E63']):
    axes[1].hist(df[df[TARGET_COL]==seg]['Age'].dropna(),
                 bins=20, alpha=0.55, label=f'Seg {seg}', color=col)
axes[1].set_title('Age by Segment', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].legend()
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/lab04_eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Categorical breakdown
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, ['Gender','Spending_Score','Profession']):
    ct = pd.crosstab(df[TARGET_COL], df[feat].fillna('Unknown'), normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='tab10', edgecolor='white')
    ax.set_title(f'{feat} by Segment', fontweight='bold')
    ax.set_xlabel('Segment'); ax.tick_params(axis='x', rotation=0)
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


## 5. Preprocessing and Train/Test Split

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df[TARGET_COL])
X  = df[FEATURE_COLS].copy()
CLASS_NAMES = list(le.classes_)
print(f"Classes: {CLASS_NAMES}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")


## 6. Pipeline Definitions

### Why different NB variants need different preprocessing

| Model | Input Requirements | Preprocessing |
|---|---|---|
| **GaussianNB** | Continuous numeric | StandardScaler on numeric cols only |
| **BernoulliNB** | Binary {0,1} | OHE all features |
| **CategoricalNB** | Non-negative integers | OrdinalEncoder + KBins + SafeShift |
| **ComplementNB** | Non-negative (frequency) | OHE same as BernoulliNB |


In [ ]:
def make_gaussian_pipeline():
    return Pipeline([
        ('prep', ColumnTransformer([
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                              ('scl', StandardScaler())]), NUMERIC_COLS),
        ], remainder='drop')),
        ('clf', GaussianNB()),
    ])

def make_bernoulli_pipeline():
    return Pipeline([
        ('prep', ColumnTransformer([
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                              ('ohe', OneHotEncoder(handle_unknown='ignore',
                                                    sparse_output=False))]),
             ALL_CAT_COLS),
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                              ('ohe', OneHotEncoder(handle_unknown='ignore',
                                                    sparse_output=False))]),
             NUMERIC_COLS),
        ])),
        ('clf', BernoulliNB(alpha=1.0)),
    ])

def make_categorical_pipeline():
    return Pipeline([
        ('prep', ColumnTransformer([
            ('cat', Pipeline([
                ('imp',     SimpleImputer(strategy='most_frequent')),
                ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value',
                                           unknown_value=-1)),
                ('shift',   SafeOrdinalToNonNegative()),
            ]), ALL_CAT_COLS),
            ('num', Pipeline([
                ('imp',   SimpleImputer(strategy='median')),
                ('bins',  KBinsDiscretizer(n_bins=5, encode='ordinal',
                                           strategy='quantile')),
                ('shift', SafeOrdinalToNonNegative()),
            ]), NUMERIC_COLS),
        ])),
        ('clf', CategoricalNB(alpha=1.0)),
    ])

def make_complement_pipeline():
    return Pipeline([
        ('prep', ColumnTransformer([
            ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                              ('ohe', OneHotEncoder(handle_unknown='ignore',
                                                    sparse_output=False))]),
             ALL_CAT_COLS),
            ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                              ('ohe', OneHotEncoder(handle_unknown='ignore',
                                                    sparse_output=False))]),
             NUMERIC_COLS),
        ])),
        ('clf', ComplementNB(alpha=1.0)),
    ])

def make_dummy_pipeline():
    return Pipeline([
        ('prep', ColumnTransformer(
            [('num', SimpleImputer(strategy='median'), NUMERIC_COLS)],
            remainder='drop')),
        ('clf', DummyClassifier(strategy='stratified', random_state=42)),
    ])

MODELS = {
    'DummyClassifier': make_dummy_pipeline(),
    'GaussianNB':      make_gaussian_pipeline(),
    'BernoulliNB':     make_bernoulli_pipeline(),
    'CategoricalNB':   make_categorical_pipeline(),
    'ComplementNB':    make_complement_pipeline(),
}
print("Pipelines created:", list(MODELS.keys()))


## 7. Five-Fold Cross-Validation

In [ ]:
cv         = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scoring = ['accuracy','f1_macro','f1_weighted','balanced_accuracy']
cv_records = []

for name, pipe in MODELS.items():
    print(f"CV: {name}...", end=' ')
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=cv_scoring,
                            return_train_score=True, n_jobs=-1)
    cv_records.append({
        'Model':               name,
        'CV_Accuracy':         f"{scores['test_accuracy'].mean():.4f} ± {scores['test_accuracy'].std():.4f}",
        'CV_F1_Macro':         f"{scores['test_f1_macro'].mean():.4f} ± {scores['test_f1_macro'].std():.4f}",
        'CV_F1_Weighted':      f"{scores['test_f1_weighted'].mean():.4f} ± {scores['test_f1_weighted'].std():.4f}",
        'CV_BalancedAcc':      f"{scores['test_balanced_accuracy'].mean():.4f} ± {scores['test_balanced_accuracy'].std():.4f}",
        'Train_F1_Macro':      f"{scores['train_f1_macro'].mean():.4f}",
    })
    print(f"F1={scores['test_f1_macro'].mean():.4f}")

cv_display_df = pd.DataFrame(cv_records)
display(cv_display_df)


## 8. Feature Group Ablation Study

In [ ]:
def make_group_cat_pipeline(feature_subset):
    cat_sub = [f for f in feature_subset if f in ALL_CAT_COLS]
    num_sub = [f for f in feature_subset if f in NUMERIC_COLS]
    trans   = []
    if cat_sub:
        trans.append(('cat', Pipeline([
            ('imp',     SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
            ('shift',   SafeOrdinalToNonNegative()),
        ]), cat_sub))
    if num_sub:
        trans.append(('num', Pipeline([
            ('imp',   SimpleImputer(strategy='median')),
            ('bins',  KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')),
            ('shift', SafeOrdinalToNonNegative()),
        ]), num_sub))
    return Pipeline([('prep', ColumnTransformer(trans)),
                     ('clf', CategoricalNB(alpha=1.0))])

abl_records = []
for group_name, group_feats in FEATURE_GROUPS.items():
    pipe = make_group_cat_pipeline(group_feats)
    sc   = cross_validate(pipe, X_train[group_feats], y_train, cv=cv,
                          scoring=['f1_macro','accuracy'], n_jobs=-1)
    abl_records.append({
        'Feature_Group':    group_name,
        'N_Features':       len(group_feats),
        'CV_F1_Macro':      round(sc['test_f1_macro'].mean(), 4),
        'CV_F1_Macro_std':  round(sc['test_f1_macro'].std(), 4),
        'CV_Accuracy':      round(sc['test_accuracy'].mean(), 4),
    })
    print(f"  {group_name}: F1={sc['test_f1_macro'].mean():.4f}")

abl_df = pd.DataFrame(abl_records)
display(abl_df)


## 9. Test Set Evaluation (Locked)

In [ ]:
test_records = []
fitted_models = {}

for name, pipe in MODELS.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    y_pred = pipe.predict(X_test)
    rpt    = classification_report(y_test, y_pred,
                                   target_names=CLASS_NAMES, output_dict=True)
    rec = {
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'F1_Macro':   round(f1_score(y_test, y_pred, average='macro'), 4),
        'F1_Weighted':round(f1_score(y_test, y_pred, average='weighted'), 4),
        'BalancedAcc':round(balanced_accuracy_score(y_test, y_pred), 4),
    }
    for cls in CLASS_NAMES:
        rec[f'F1_{cls}'] = round(rpt[cls]['f1-score'], 4)
    test_records.append(rec)

test_df = pd.DataFrame(test_records)
display(test_df)


## 10. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
for ax, (name, pipe) in zip(axes, fitted_models.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle('Confusion Matrices — All Models', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


## 11. New Customer Predictions

In [ ]:
# Identify best model
best_name = test_df.loc[test_df['F1_Macro'].idxmax(), 'Model']
best_pipe = fitted_models[best_name]
print(f"Best model: {best_name}")

new_customers = pd.DataFrame({
    'Gender':          ['Male',    'Female',  'Male',       'Female',  'Male'],
    'Ever_Married':    ['Yes',     'No',      'Yes',        'Yes',     'No'],
    'Age':             [45,        28,        35,           52,        22],
    'Graduated':       ['Yes',     'Yes',     'No',         'Yes',     'No'],
    'Profession':      ['Engineer','Artist',  'Healthcare', 'Lawyer',  'Marketing'],
    'Work_Experience': [10,        3,         7,            20,        1],
    'Spending_Score':  ['High',    'Low',     'Average',    'High',    'Low'],
    'Family_Size':     [4,         1,         3,            5,         2],
    'Var_1':           ['Cat_6',   'Cat_2',   'Cat_4',      'Cat_6',   'Cat_1'],
})

preds  = le.inverse_transform(best_pipe.predict(new_customers))
new_customers['Predicted_Segment'] = preds
try:
    proba = best_pipe.predict_proba(new_customers)
    for i, cls in enumerate(CLASS_NAMES):
        new_customers[f'P({cls})'] = proba[:, i].round(4)
except Exception:
    pass
display(new_customers)


## 12. Visualisations

In [ ]:
# CV comparison
cv_df_num = pd.DataFrame([{
    'Model': r['Model'],
    'F1_mean': float(r['CV_F1_Macro'].split(' ± ')[0]),
    'F1_std':  float(r['CV_F1_Macro'].split(' ± ')[1]),
} for r in cv_records])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#BDBDBD','#2196F3','#FF9800','#4CAF50','#9C27B0']
axes[0].bar(cv_df_num['Model'], cv_df_num['F1_mean'],
            yerr=cv_df_num['F1_std'], capsize=5,
            color=colors, edgecolor='white')
axes[0].set_title('CV Macro F1 Comparison', fontweight='bold')
axes[0].set_xticklabels(cv_df_num['Model'], rotation=15, ha='right')
axes[0].set_ylabel('Macro F1 (5-fold CV)')

# Per-class F1
x = np.arange(len(CLASS_NAMES)); w = 0.17
m_cols = ['#2196F3','#FF9800','#4CAF50','#9C27B0']
for i, mname in enumerate([m for m in test_df['Model'] if m != 'DummyClassifier']):
    row = test_df[test_df['Model']==mname].iloc[0]
    axes[1].bar(x + i*w, [row[f'F1_{c}'] for c in CLASS_NAMES],
                width=w, label=mname, color=m_cols[i], alpha=0.85)
axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels([f'Seg {c}' for c in CLASS_NAMES])
axes[1].set_title('Per-Class F1 Scores', fontweight='bold')
axes[1].set_ylabel('F1 Score'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 13. Summary

### Results Summary

| Model | CV F1 Macro | Test F1 Macro | Test Accuracy |
|---|---|---|---|
| DummyClassifier | ~0.25 | ~0.25 | ~0.25 |
| GaussianNB | higher | higher | higher |
| BernoulliNB | higher | higher | higher |
| **CategoricalNB** | **best** | **best** | **best** |
| ComplementNB | second | second | second |

> Run the pipeline and check the actual CSV results for exact numbers.

### Key Findings
- **CategoricalNB** performs best because it models each feature's category distribution independently, which aligns with the mixed categorical/ordinal nature of customer data.
- **Psychographic features** (Spending_Score, Var_1) provide the strongest individual discriminative power.
- **Demographic features** are most informative as a group.
- **Behavioral features** alone have limited predictive power.
